# Verification checks and an input-range warning

**Optional code walkthrough for AI 101 (Module 15).** Not graded. Quiz scores do not depend on it.

**Objective.** Write a small automatic known-case test and use it to catch a unit bug. Then check whether a new operating point is covered by the training conditions, and raise a warning when recent inputs leave the development range.

**Supports.** Lesson 15.1 (sections “Operating coverage is more than input bounds” and “Distribution shift and drift”) and Lesson 15.2 (sections “Verification: did we implement the method correctly?” and “Validation: does it represent the real system?”).

Run the cells from top to bottom. All data are synthetic and invented for teaching.


## About the data

The first example uses the pump power relationship of Lesson 15.2, P = Δp × Q, with the values of the Module 2 example: 100 kPa and 0.02 m³/s give 2,000 W.

The second and third examples use invented pump operating conditions: flow rate Q in m³/s and pressure rise Δp in kPa. All data are synthetic and are not from real equipment.

```
Training operating points (synthetic): Q (m³/s), delta p (kPa)
0.010   60
0.012   70
0.020   95
0.028  120
0.030  130
```


## Step 1. A known-case test

Lesson 15.2 says verification uses known cases, dimensional analysis and bounds. Here you write the function `check`. It compares a result with an expected value and prints PASS or FAIL. Then you test a hydraulic power function with a known case and with a scaling rule.

In [ ]:
# Step 1: verification with known cases
def check(name, got, expected):
    if abs(got - expected) < 1e-9:
        print(f"PASS: {name}")
    else:
        print(f"FAIL: {name} (got {got}, expected {expected})")

def hydraulic_power(dp_kpa, q):
    return dp_kpa * 1000 * q      # kPa -> Pa, then Pa * m³/s = W

check("100 kPa and 0.02 m³/s give 2000 W", hydraulic_power(100, 0.02), 2000)
check("doubling the flow doubles the power", hydraulic_power(100, 0.04), 2 * hydraulic_power(100, 0.02))

**Expected output**

```
PASS: 100 kPa and 0.02 m³/s give 2000 W
PASS: doubling the flow doubles the power
```

Both tests pass. The first is a known case from the Module 2 example. The second is a scaling rule: power is proportional to flow. Passing shows that the function implements the intended equation. It does not show that the equation represents a real pump.

## Step 2. Catch a unit bug

Now a version of the function forgets the conversion from kilopascals to pascals. The same known case should reveal the bug.

In [ ]:
# Step 2: the known case catches a bug
def hydraulic_power_buggy(dp_kpa, q):
    return dp_kpa * q             # forgot kPa -> Pa

check("buggy function, known case", hydraulic_power_buggy(100, 0.02), 2000)

**Expected output**

```
FAIL: buggy function, known case (got 2.0, expected 2000)
```

The test fails because the result is 1,000 times too small. Without the known case, the bug could have stayed hidden. The same function would also give the same wrong answer every time. Reproducibility is not correctness (Lesson 15.2).

## Step 3. Is a new point covered by the training data?

Lesson 15.1 says that a point can be inside the minimum and maximum of each input and still be unrepresentative. Here you check two things for each new point: whether each input is within its training range, and how many training points lie nearby. “Nearby” here means within 0.005 m³/s and within 10 kPa. This closeness rule is an invented example, not a standard.

In [ ]:
# Step 3: inside each input range, but is the combination covered?
train_points = [(0.010, 60), (0.012, 70), (0.020, 95), (0.028, 120), (0.030, 130)]
q_values = []
dp_values = []
for q, dp in train_points:
    q_values.append(q)
    dp_values.append(dp)

new_points = [("P1", 0.011, 65), ("P2", 0.030, 60)]
for name, q, dp in new_points:
    inside = (min(q_values) <= q and q <= max(q_values)
              and min(dp_values) <= dp and dp <= max(dp_values))
    near = 0
    for train_q, train_dp in train_points:
        if abs(q - train_q) <= 0.005 and abs(dp - train_dp) <= 10:
            near += 1
    if inside and near > 0:
        print(f"{name}: inside each input range; {near} nearby training points -> covered")
    elif inside:
        print(f"{name}: inside each input range; {near} nearby training points -> not covered, treat as extrapolation")
    else:
        print(f"{name}: outside an input range -> extrapolation")

**Expected output**

```
P1: inside each input range; 2 nearby training points -> covered
P2: inside each input range; 0 nearby training points -> not covered, treat as extrapolation
```

Both points are inside the training range of each input. But P2 combines the largest flow with the smallest pressure rise. No training point is near that combination. A check of each input alone would have accepted P2. Define the operating domain by the combinations that were actually observed.

## Step 4. Warn when recent inputs leave the development range

Lesson 15.1 says to monitor input ranges and compare them with the development conditions. An input change is a warning signal. It is not proof that accuracy has dropped. Here you count recent flow readings that lie outside the development range of 0.010 to 0.030 m³/s.

In [ ]:
# Step 4: a simple input-range monitor
dev_min = 0.010           # m³/s, from the development data
dev_max = 0.030           # m³/s
recent = [0.012, 0.026, 0.034, 0.036]    # new flow readings, synthetic

outside = 0
for q in recent:
    if q < dev_min or q > dev_max:
        outside += 1

print(f"Readings outside the development range: {outside} of {len(recent)}")
if outside > 0:
    print("Warning signal: the inputs have moved. This does not prove the accuracy has dropped.")
    print("Next: check outcomes when trustworthy labels are available.")

**Expected output**

```
Readings outside the development range: 2 of 4
Warning signal: the inputs have moved. This does not prove the accuracy has dropped.
Next: check outcomes when trustworthy labels are available.
```

Two of the four readings are outside the range seen in development. The model may now be extrapolating. The check does not say whether the predictions are wrong. It tells you to look. Decide the action rules in advance, as the lesson says.

## Try it

Change a value and observe. In Step 4, change the list to `recent = [0.012, 0.026, 0.024, 0.036]`. Then run the cell again.

**Expected result**

```
Readings outside the development range: 1 of 4
Warning signal: the inputs have moved. This does not prove the accuracy has dropped.
Next: check outcomes when trustworthy labels are available.
```

Only one reading is now outside the range, and the warning still appears. Any reading outside the range raises the warning. A real monitor would also use a rule for how many readings and how long, and that rule must be decided in advance.

## Limits

- The data are synthetic. The coverage rule in Step 3 and the range check in Step 4 are simple teaching examples, not standards.
- Passing a known-case test is verification. It is not validation. Validation needs independent, calibrated measurements from the real system (Lesson 15.2).
- The code does not measure calibration, drift in accuracy or uncertainty. It only checks implementation and input ranges.
- Real operating domains have more than two inputs, and the closeness rule would need justification for each.

Lab 4 (Module 14) uses known cases and unit checks on a supplied draft. This walkthrough adds coverage and input-range warnings.